# Train và so sánh PPO, MPC-DAgger, MPC-Inject (Colab, GPU)

Notebook này dùng lệnh **`train` chuẩn của mjlab** (log TensorBoard, lưu checkpoint), train
lần lượt 3 phương pháp trên cùng task và seed, rồi so sánh đường học và xem video policy.
Mặc định train như tài liệu mjlab (4096 env, 500 vòng); có thể chọn thiết lập giống các thí
nghiệm trong báo cáo (64 env, 200 vòng) ở mục 0.

| Phương pháp | Cách dùng MPC |
|---|---|
| **PPO** | Không dùng MPC (mốc so sánh) |
| **MPC-DAgger** | 8 env phụ do policy tự chạy; MPOPI (K = 8 × 4 vòng, H = 20) gắn nhãn trạng thái policy đi qua; loss bắt chước giảm từ 1 về 0 sau 30 vòng; gắn nhãn trong 20 vòng đầu |
| **MPC-Inject** | MPOPI tự chạy 8 env phụ trong 20 vòng đầu (không nhiễu), giữ cố định 2560 mẫu; **mọi** batch PPO có 25% là mẫu MPC, coi như dữ liệu của chính PPO |

Khác với notebook benchmark, ở đây đường học là **reward lúc train** (policy có nhiễu khám
phá, `Train/mean_reward`), không phải điểm đánh giá tất định. Dùng để quan sát và xem
video; kết luận thống kê vẫn dựa trên notebook benchmark.

**Lưu ý:** trên Swingup, MPC-Inject hay bị lỗi giữa chừng
(`normal expects all elements of std >= 0.0`, độ lệch chuẩn của policy bị đẩy xuống âm).
Nếu một lần chạy bị lỗi, các lần chạy khác vẫn tiếp tục.

**Trước khi chạy:** *Runtime → Change runtime type → GPU* (T4 là đủ cho Cartpole).

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpc-stage1"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
TASK = "Mjlab-Cartpole-Swingup"  #@param ["Mjlab-Cartpole-Swingup", "Mjlab-Cartpole-Balance"]
#@markdown **Thiết lập train:**
#@markdown - `mặc định mjlab`: như tài liệu mjlab (4096 env, 500 vòng); MPC dùng 64 env × 32 bước
#@markdown   để mẫu MPC vẫn đủ chiếm 25% batch của MPC-Inject.
#@markdown - `giống thí nghiệm`: 64 env, 200 vòng, MPC 8 env × 16 bước, như các kết quả trong báo cáo.
PRESET = "mặc định mjlab"  #@param ["mặc định mjlab", "giống thí nghiệm"]
if PRESET == "mặc định mjlab":
  NUM_ENVS, ITERATIONS, MPC_ENVS, MPC_STEPS = 4096, 500, 64, 32
else:
  NUM_ENVS, ITERATIONS, MPC_ENVS, MPC_STEPS = 64, 200, 8, 16
#@markdown Seed, cách nhau bằng dấu cách:
SEEDS = "1 2 3"  #@param {type:"string"}
#@markdown Phương pháp cần train:
TRAIN_PPO = True  #@param {type:"boolean"}
TRAIN_DAGGER = True  #@param {type:"boolean"}
TRAIN_INJECT = True  #@param {type:"boolean"}
INJECT_FRACTION = 0.25  #@param {type:"number"}
#@markdown Số lần train chạy song song trên GPU:
PARALLEL = 2  #@param {type:"integer"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  OUT_ROOT = "/content/drive/MyDrive/mpc_train_compare"
else:
  OUT_ROOT = "/content/mpc_train_compare"
LOG_ROOT = f"{OUT_ROOT}/train_logs"
CONSOLE_DIR = f"{OUT_ROOT}/train_console"
os.makedirs(CONSOLE_DIR, exist_ok=True)
SEED_LIST = [int(s) for s in SEEDS.split()]
print("Kết quả lưu tại:", OUT_ROOT, "| seed:", SEED_LIST)
print(f"{PRESET}: PPO {NUM_ENVS} env × {ITERATIONS} vòng; MPC {MPC_ENVS} env × {MPC_STEPS} bước "
      f"trong 20 vòng đầu = {MPC_ENVS * MPC_STEPS * 20} mẫu MPC")

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import subprocess


def run(cmd: str, cwd: str | None = None) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

Nhánh `mpc-stage1` phải được push bản mới nhất (có `inject_fraction`).

In [ ]:
import shutil

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
config_src = open(f"{REPO_DIR}/src/mjlab/rl/mpopi/config.py").read()
if "inject_fraction" not in config_src:
  raise RuntimeError(
    "Code trên GitHub đã cũ (thiếu inject_fraction). "
    "Hãy push bản mới nhất của nhánh mpc-stage1 rồi chạy lại ô này."
  )
print("OK: code có MPC-DAgger và MPC-Inject.")

## 3. Cài đặt môi trường

`uv sync` **không** kèm `--extra cpu` để có PyTorch bản CUDA. Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch; print('torch', torch.__version__, 'cuda', "
  "torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)

## 4. Lệnh train của từng phương pháp

Mọi phương pháp dùng cùng lệnh `uv run train`, chỉ khác các tùy chọn
`--agent.algorithm.mpopi...`. Ô này in ra lệnh đầy đủ để bạn có thể chạy tay.

In [ ]:
A = "--agent.algorithm.mpopi"
MPC_TEACHER = (
  f"{A}.mode mpc_ppo {A}.mpc.num-envs {MPC_ENVS} {A}.mpc.num-steps {MPC_STEPS} "
  f"{A}.mpc.collect-iterations 20 {A}.mpc.execution-std 0 "
  f"{A}.mpc.planner.num-samples 8 {A}.mpc.planner.iterations 4 {A}.mpc.planner.horizon 20"
)
METHODS = {}
if TRAIN_PPO:
  METHODS["PPO"] = f"{A}.mode ppo"
if TRAIN_DAGGER:
  METHODS["MPC-DAgger"] = (
    f"{MPC_TEACHER} {A}.mpc.use-in-ppo False {A}.mpc.driver policy "
    f"{A}.mpc.bc-coef 1.0 {A}.mpc.bc-iterations 30 {A}.mpc.max-age 10 {A}.mpc.buffer-segments 8"
  )
if TRAIN_INJECT:
  METHODS["MPC-Inject"] = (
    f"{MPC_TEACHER} {A}.mpc.correction False {A}.mpc.bc-coef 0 "
    f"{A}.mpc.inject-fraction {INJECT_FRACTION} {A}.mpc.max-age None {A}.mpc.buffer-segments 20"
  )


def train_cmd(method: str, seed: int) -> str:
  return (
    f"uv run train {TASK} --agent.logger tensorboard --env.scene.num-envs {NUM_ENVS} "
    f"--agent.max-iterations {ITERATIONS} --agent.seed {seed} "
    f"--agent.run-name {method}_s{seed} --log-root {LOG_ROOT} {METHODS[method]}"
  )


for m in METHODS:
  print(f"# {m}\n{train_cmd(m, SEED_LIST[0])}\n")

## 5. Chạy thử nhanh (3 vòng mỗi phương pháp)

Kiểm tra cả 3 phương pháp chạy được trên GPU trước khi train dài. Nếu ô này lỗi thì dừng lại.

In [ ]:
import time

for m in METHODS:
  t0 = time.time()
  cmd = train_cmd(m, 0).replace(f"--agent.max-iterations {ITERATIONS}", "--agent.max-iterations 3")
  cmd = cmd.replace(f"--log-root {LOG_ROOT}", f"--log-root {OUT_ROOT}/smoke_logs")
  cmd = cmd.replace("collect-iterations 20", "collect-iterations 2")
  run(f"{cmd} > {OUT_ROOT}/smoke_{m}.log 2>&1 || (tail -30 {OUT_ROOT}/smoke_{m}.log; exit 1)", cwd=REPO_DIR)
  print(f"OK {m}: {time.time() - t0:.0f} s")

## 6. Train (chạy nền)

Chạy nền để bạn xem tiến độ (mục 7) và TensorBoard (mục 8) trong lúc train. Mỗi lần train
xong sẽ để lại file `.done`; chạy lại ô này sau khi Colab ngắt thì các lần đã xong được bỏ qua.

In [ ]:
import shlex
import signal

if "TRAIN_PROCS" in globals() and any(p.poll() is None for p in TRAIN_PROCS):
  raise RuntimeError("Đang có train chạy nền. Dừng ở mục 7.2 trước.")

jobs = [(m, s) for s in SEED_LIST for m in METHODS]
lanes = [[] for _ in range(max(1, PARALLEL))]
for i, (m, s) in enumerate(jobs):
  name = f"{m}_s{s}"
  log = shlex.quote(f"{CONSOLE_DIR}/{name}.log")
  done = shlex.quote(f"{CONSOLE_DIR}/{name}.done")
  lanes[i % len(lanes)].append(
    f"[ -f {done} ] || ({train_cmd(m, s)} > {log} 2>&1 && touch {done})"
  )
env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
TRAIN_PROCS = [
  subprocess.Popen(["bash", "-c", "; ".join(lane)], cwd=REPO_DIR, env=env, start_new_session=True)
  for lane in lanes if lane
]
print(f"Đã bắt đầu {len(jobs)} lần train trên {len(TRAIN_PROCS)} luồng song song:")
for m, s in jobs:
  print(f"  {m} seed {s}")

### 7.1 Xem tiến độ

Chạy lại ô này bất cứ lúc nào: in dòng tiến độ mới nhất của từng lần train và vẽ
`Mean reward` theo vòng lặp.

In [ ]:
#@title Xem tiến độ
import glob
import re

import matplotlib.pyplot as plt

ANSI = re.compile(r"\x1b\[[0-9;]*m")


def parse_log(path):
  text = ANSI.sub("", open(path, errors="ignore").read())
  curve = []
  for block in text.split("Learning iteration")[1:]:
    it = re.match(r"\s*(\d+)/", block)
    rew = re.search(r"Mean reward:\s*(-?[\d.]+)", block)
    if it and rew:
      curve.append((int(it.group(1)), float(rew.group(1))))
  err = re.findall(r"^\w*Error: .*$", text, flags=re.M)
  return curve, err[-1] if err else None


COLORS = {"PPO": "#555555", "MPC-DAgger": "#d62728", "MPC-Inject": "#1f77b4"}
fig, ax = plt.subplots(figsize=(9, 4))
for path in sorted(glob.glob(f"{CONSOLE_DIR}/*.log")):
  name = os.path.basename(path).removesuffix(".log")
  method = name.rsplit("_s", 1)[0]
  curve, err = parse_log(path)
  done = os.path.exists(path.replace(".log", ".done"))
  status = "XONG" if done else ("LỖI: " + err if err else "đang chạy / khởi động")
  last = f"vòng {curve[-1][0]}, mean reward {curve[-1][1]:.2f}" if curve else "chưa có vòng nào"
  print(f"{name:>16}: {last} | {status}")
  if curve:
    xs, ys = zip(*curve)
    ax.plot(xs, ys, color=COLORS.get(method, None), alpha=0.8, label=name)
running = "TRAIN_PROCS" in globals() and any(p.poll() is None for p in TRAIN_PROCS)
print("\nTrạng thái:", "ĐANG TRAIN" if running else "không có train nào đang chạy")
ax.set_xlabel("Vòng lặp")
ax.set_ylabel("Mean reward (lúc train)")
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=3)
plt.show()

### 7.2 Chờ train xong / dừng train

In [ ]:
#@title Chờ train xong (in tiến độ mỗi 5 phút; bấm Stop để ngừng chờ, train vẫn chạy)
while "TRAIN_PROCS" in globals() and any(p.poll() is None for p in TRAIN_PROCS):
  time.sleep(300)
  n_done = len(glob.glob(f"{CONSOLE_DIR}/*.done"))
  print(time.strftime("%H:%M"), f"{n_done} lần train đã xong")
print("Train đã kết thúc.")

In [ ]:
#@title Dừng hẳn train
for p in TRAIN_PROCS if "TRAIN_PROCS" in globals() else []:
  if p.poll() is None:
    os.killpg(p.pid, signal.SIGTERM)
print("Đã gửi lệnh dừng.")

## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$LOG_ROOT" --reload_interval 30

## 9. So sánh 3 phương pháp

Đọc log TensorBoard của mọi lần train, lấy trung bình theo seed. Bảng in ra: reward trung
bình suốt quá trình train (tương tự AUC), reward 10% vòng cuối, độ dài episode cuối, số lần
train bị lỗi.

In [ ]:
import numpy as np
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def run_dir(method, seed):
  found = glob.glob(f"{LOG_ROOT}/**/*_{method}_s{seed}", recursive=True)
  return max(found, key=os.path.getmtime) if found else None


def scalars(path, tag):
  ea = EventAccumulator(path, size_guidance={"scalars": 0})
  ea.Reload()
  if tag not in ea.Tags()["scalars"]:
    return {}
  return {e.step: e.value for e in ea.Scalars(tag)}


data = {}
for m in METHODS:
  for s in SEED_LIST:
    d = run_dir(m, s)
    if d:
      data[(m, s)] = {
        "reward": scalars(d, "Train/mean_reward"),
        "length": scalars(d, "Train/mean_episode_length"),
        "done": os.path.exists(f"{CONSOLE_DIR}/{m}_s{s}.done"),
      }

print(f"{'phương pháp':>12} {'xong':>6} {'reward TB':>10} {'reward cuối':>12} {'độ dài cuối':>12}")
for m in METHODS:
  runs = [data[(m, s)] for s in SEED_LIST if (m, s) in data and data[(m, s)]["reward"]]
  if not runs:
    print(f"{m:>12}: chưa có dữ liệu")
    continue
  n_done = sum(r["done"] for r in runs)
  mean_r = np.mean([np.mean(list(r["reward"].values())) for r in runs])
  def tail(d):
    v = [d[k] for k in sorted(d)]
    return np.mean(v[-max(1, len(v) // 10):])
  print(f"{m:>12} {n_done:>3}/{len(SEED_LIST):<2} {mean_r:10.2f} "
        f"{np.mean([tail(r['reward']) for r in runs]):12.2f} {np.mean([tail(r['length']) for r in runs]):12.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for m in METHODS:
  runs = [data[(m, s)] for s in SEED_LIST if (m, s) in data and data[(m, s)]["reward"]]
  for ax, key, label in ((axes[0], "reward", "Mean reward"), (axes[1], "length", "Mean episode length")):
    for i, r in enumerate(runs):
      steps = sorted(r[key])
      ax.plot(steps, [r[key][k] for k in steps], color=COLORS[m], alpha=0.25, lw=1)
    if runs:
      common = sorted(set.intersection(*(set(r[key]) for r in runs)))
      ax.plot(common, [np.mean([r[key][k] for r in runs]) for k in common], color=COLORS[m], lw=2.5,
              label=f"{m} (TB {len(runs)} seed)")
    ax.set_xlabel("Vòng lặp")
    ax.set_ylabel(label + " (lúc train)")
    ax.grid(alpha=0.3)
axes[0].axvspan(0, 20, color="#fde0c5", alpha=0.4, lw=0)
axes[0].legend()
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/train_compare.png", dpi=150)
plt.show()

## 10. Video policy của từng phương pháp

Quay video policy đã train (checkpoint mới nhất của seed đầu tiên) bằng lệnh `play`.

In [ ]:
#@title Quay video
VIDEO_STEPS = 400  #@param {type:"integer"}
VIDEO_SEED = SEED_LIST[0]
from IPython.display import HTML, Video, display

for m in METHODS:
  d = run_dir(m, VIDEO_SEED)
  ckpts = glob.glob(f"{d}/model_*.pt") if d else []
  if not ckpts:
    print(f"{m}: không có checkpoint")
    continue
  ckpt = max(ckpts, key=lambda p: int(re.search(r"model_(\d+)", p).group(1)))
  video = os.path.join(os.path.dirname(ckpt), "videos/play/rl-video-step-0.mp4")
  if os.path.exists(video):
    os.remove(video)
  # play keeps a web viewer open after recording, so run it in the background.
  proc = subprocess.Popen(
    f"uv run play {TASK} --checkpoint-file {ckpt} --video True "
    f"--video-length {VIDEO_STEPS} --viewer viser",
    shell=True, cwd=REPO_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    start_new_session=True,
  )
  start, last_size = time.time(), -1
  while time.time() - start < 600:
    time.sleep(5)
    if os.path.exists(video):
      size = os.path.getsize(video)
      if size == last_size and size > 0:
        break
      last_size = size
  os.killpg(proc.pid, signal.SIGTERM)
  if os.path.exists(video):
    display(HTML(f"<h3>{m} (seed {VIDEO_SEED}, {os.path.basename(ckpt)})</h3>"))
    display(Video(video, embed=True, width=480))
  else:
    print(f"{m}: không tạo được video")

## 11. Tải kết quả về

In [ ]:
shutil.make_archive("/content/mpc_train_compare_results", "zip", OUT_ROOT)
from google.colab import files

files.download("/content/mpc_train_compare_results.zip")